> 在线讲义：[第 42 章 · 参数化与初始化：标准参数化 vs μP，超参从小模型迁移](https://wikieden.github.io/math-and-eval-stats/lessons/42-mup.html) · 本书《数学与实验统计》

# 第 42 章 · 参数化与初始化：标准参数化 vs μP，超参从小模型迁移

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/42-mup.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 初始化 | 1/√n | `init_std(n, "hidden", "sp")` | 0.125 … 0.0156 |
| 第二步 一步更新 | lr·Σ\|x\| | `hidden_update_size(n, mode=...)` | SP 0.048 → 3.28；μP ≈ 0.05 |
| 第三步 输出层 | 1/n 初始化 | `output_logit_scale(n, mode)` | SP ≈ 1；μP 0.132 → 0.016 |

例子：一个隐藏层 h = W x（宽度 n 从 64 到 4096），输入每个分量 O(1)。看一步 Adam 式更新（每个权重变化 ±lr）后，
h 的每个分量变化多大：标准参数化（SP）随宽度线性涨，μP 把隐藏层学习率缩放成 1/n 后不随宽度变。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
WIDTHS = (64, 256, 1024, 4096)

**函数 `init_std`**：SP：所有层 1/√fan_in。μP（Tensor Programs V 表 3 的 Adam 版本之一）：输入层、隐藏层 1/√fan_in，输出层 1/fan_in。

In [ ]:
def init_std(n: int, layer: str, mode: str) -> float:
    """SP：所有层 1/√fan_in。μP（Tensor Programs V 表 3 的 Adam 版本之一）：输入层、隐藏层 1/√fan_in，输出层 1/fan_in。"""
    if mode == "mup" and layer == "out":
        return 1.0 / n
    return 1.0 / np.sqrt(n)

**函数 `hidden_update_size`**：Adam 的一步更新近似为 ΔW = −lr·sign(G)；梯度 G = g xᵀ（秩 1），所以 ΔW 的每一行和 x 的符号对齐。

In [ ]:
def hidden_update_size(n: int, lr: float = 1e-3, mode: str = "sp", seed: int = 0) -> float:
    """Adam 的一步更新近似为 ΔW = −lr·sign(G)；梯度 G = g xᵀ（秩 1），所以 ΔW 的每一行和 x 的符号对齐。
    Δh = ΔW x 的每个分量 ≈ lr·Σ|x_j| ∝ lr·n。μP 把隐藏层学习率设为 lr·(n0/n)（n0 = 64）。返回 Δh 分量的均方根。"""
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(n)
    g = rng.standard_normal(n)
    eta = lr if mode == "sp" else lr * 64 / n
    dw = -eta * np.sign(np.outer(g, x))
    return float(np.sqrt(((dw @ x) ** 2).mean()))

**函数 `output_logit_scale`**：初始化时输出 logit = w_outᵀ h（h 每个分量 O(1)）的大小：SP 是 O(1)，μP 输出层 1/n 初始化后是 O(1/√n)，宽了以后从 0 附近起步。

In [ ]:
def output_logit_scale(n: int, mode: str, seed: int = 0) -> float:
    """初始化时输出 logit = w_outᵀ h（h 每个分量 O(1)）的大小：SP 是 O(1)，μP 输出层 1/n 初始化后是 O(1/√n)，宽了以后从 0 附近起步。"""
    rng = np.random.default_rng(seed)
    h = rng.standard_normal((2000, n))
    w = rng.normal(0, init_std(n, "out", mode), n)
    return float((h @ w).std())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 初始化只管前向：两种参数化都让隐藏层激活 O(1)")
for n in WIDTHS:
    print(f"  宽度 {n}：隐藏层初始化标准差 1/√n = {init_std(n, 'hidden', 'sp'):.4f}")

In [ ]:
print("\n第二步 学习率才是问题：同样的 lr，宽度越大一步改得越多")
for n in WIDTHS:
    print(f"  宽度 {n}：SP 一步后 Δh 分量 {hidden_update_size(n):.4f}；μP（lr × 64/n）{hidden_update_size(n, mode='mup'):.4f}")

In [ ]:
print("\n第三步 输出层：μP 用 1/n 初始化")
for n in WIDTHS:
    print(f"  宽度 {n}：初始 logit 标准差 SP {output_logit_scale(n, 'sp'):.3f}，μP {output_logit_scale(n, 'mup'):.4f}")

In [ ]:
print("\n第四步 超参迁移：在宽 256 上调好 lr，μP 下直接用到宽 4096")
print("  SP：最优 lr 随宽度大约按 1/n 变小，换宽度要重调；μP：隐藏层 lr 已经按 1/n 缩放，名义 lr 不变")

## 3. 代码里才看得到的

1. **用符号更新近似 Adam**：Adam 在梯度方向稳定时更新幅度接近 lr·sign(g)，正好暴露“同号相加”的问题；SGD 的情形结论不同（SGD 的梯度本身就随宽度缩放）。
2. **以宽 64 为基准**：μP 只规定“lr ∝ 1/n”，常数要在某个基准宽度上调出来，代码取 64。
3. **第四步没有代码**：真正的 μTransfer 要在小模型上扫 lr、训练到收敛，再在大模型上验证，留给自己做。

## 4. 自己改一改

1. 把更新换成 SGD 式的 ΔW = −lr·g xᵀ（不取符号），SP 下 Δh 还随宽度线性涨吗？
2. 把 x 换成稀疏的（大部分分量是 0），同号相加的效应会变弱多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 初始化管前向，1/√n 让随机项抵消；
> 更新同号会相加，宽度翻倍一步也翻倍；
> μP 隐藏层 lr 除以 n，输出层用 1/n 初始化；
> 小模型调好的超参，原样搬到大模型。